# 04 — Canonical Gold referral model and snapshots

Build Gold referral facts and KPI views from the current Silver state. When
`AS_OF_DATE` is supplied by the archive replay notebook, calculations and the
snapshot use that historical export date. With a blank parameter, the notebook
uses the current date for the live pipeline.


In [ ]:
AS_OF_DATE = ""  # Optional YYYY-MM-DD; archive replay passes the month-end export date.
GOLD_SCHEMA = "gold"
SNAPSHOT_TABLE = "gold.fact_referral_snapshot"

JOB_RUN_ID = ""  # Parent orchestration correlation ID.


In [ ]:
# 90_run_live_pipeline executes 00_setup_cfg before this child notebook.


In [ ]:
from datetime import date, datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F

if AS_OF_DATE:
    AS_OF_DATE_VALUE = datetime.strptime(AS_OF_DATE, "%Y-%m-%d").date()
else:
    AS_OF_DATE_VALUE = date.today()
AS_OF_SQL = f"DATE '{AS_OF_DATE_VALUE.isoformat()}'"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print(f"Gold as-of date: {AS_OF_DATE_VALUE}")


In [ ]:
GOLD_SOURCE_REQUIREMENTS = {
    "silver.referral": {
        "referral_id", "required_start_date", "response_required_by_date",
        "placement_type", "referral_created_date", "referral_modified_date",
        "referral_status", "export_date",
    },
    "silver.offer": {
        "offer_id", "referral_provider_id", "offer_status", "provider_home_id",
        "offer_date", "last_modified_date", "offer_type",
        "estimated_start_date", "core_weekly_fee", "education_weekly_fee",
        "decline_reason_other", "decline_reason", "withdraw_reason",
        "child_summary_needs", "export_date",
    },
    "silver.referral_provider": {
        "referral_provider_id", "referral_id", "provider_id", "export_date",
        "is_excluded", "is_declined", "is_cancelled", "is_closed",
    },
    "silver.ipa": {
        "referral_id", "created_datetime", "updated_datetime",
        "ipa_id", "offer_id", "placement_admission_date",
        "costs_total_weekly_fee", "status", "closed", "closed_datetime",
        "export_date",
    },
    "silver.referral_person": {
        "person_id", "referral_id", "child_index",
    },
    "silver.referral_closure_reason_summary": {
        "referral_id", "closed_referral_reason_bucket",
    },
    "silver.referral_lifecycle_event": {
        "event_id", "referral_id", "event_type", "event_timestamp",
        "sequence_number", "created_by", "created_timestamp",
    },
    "silver.referral_enrichment": {
        "referral_id", "cnt_offer_made", "first_action_date",
        "unique_homes_offered", "estimated_weekly_cost",
        "first_offer_date", "offer_accepted_date", "ipa_issued_date",
        "referral_closed_date", "last_activity_date",
        "first_provider_seen_date", "is_not_seen_by_providers",
        "ipa_placement_admission_date", "ipa_2_signatures",
        "ipa_last_signature_date", "ipa_due_diligence_min_review_date",
    },
}

gold_input_issues = []
for table_name, required_columns in GOLD_SOURCE_REQUIREMENTS.items():
    if not spark.catalog.tableExists(table_name):
        gold_input_issues.append(f"{table_name}: table is missing")
        continue
    actual_columns = {field.name.lower() for field in spark.table(table_name).schema.fields}
    missing_columns = sorted(required_columns - actual_columns)
    if missing_columns:
        gold_input_issues.append(f"{table_name}: missing {missing_columns}")

if gold_input_issues:
    raise RuntimeError(
        "Gold source validation failed. "
        + "; ".join(gold_input_issues)
        + ". Deploy the current setup notebook so monitoring.cfg_schema_contract_column is refreshed, then rerun "
          "Silver for this snapshot before 04_gold_model."
    )

print("Gold source validation passed")
# Lifecycle events are an explicit Silver derivation from referral, offer, and
# IPA timestamps; they are not a source-system referral-event audit log.
EVENT_ROLLUP_SOURCE = "silver.referral_lifecycle_event"



In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW gold.fact_referral AS
WITH referral_history AS (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY referral_id
    ORDER BY COALESCE(referral_modified_date, referral_created_date, export_date) DESC,
             export_date DESC
  ) AS row_number_current
  FROM silver.referral
),
referral_current AS (
  SELECT * FROM referral_history WHERE row_number_current = 1
),
referral_created AS (
  SELECT referral_id, MIN(referral_created_date) AS ReferralCreatedDate
  FROM silver.referral GROUP BY referral_id
),
referral_enrichment AS (
  SELECT * FROM silver.referral_enrichment
),
referral_child AS (
  SELECT referral_id, MIN(CAST(person_id AS STRING)) AS ChildID
  FROM silver.referral_person
  GROUP BY referral_id
),
closure_reason AS (
  SELECT referral_id, closed_referral_reason_bucket AS ReferralClosureReason
  FROM silver.referral_closure_reason_summary
),
base AS (
  SELECT r.referral_id AS ReferralID, child.ChildID, c.ReferralCreatedDate,
    r.required_start_date AS RequiredPlacementDate,
    r.response_required_by_date AS ResponseRequiredDate,
    r.referral_modified_date AS ReferralModifiedTimestamp,
    r.referral_status AS CurrentStatus, r.placement_type AS PlacementTypeRequired,
    x.first_action_date AS FirstActionDate, x.first_offer_date AS FirstOfferDate,
    x.offer_accepted_date AS OfferAcceptedDate, x.ipa_issued_date AS IPAIssuedDate,
    x.referral_closed_date AS ReferralClosedDate,
    closure.ReferralClosureReason,
    x.last_activity_date AS LastActivityDate,
    COALESCE(x.cnt_offer_made, 0) AS CntOfferMade,
    x.first_provider_seen_date AS FirstProviderSeenDate,
    x.is_not_seen_by_providers AS IsNotSeenByProviders,
    x.ipa_placement_admission_date AS IPAPlacementAdmissionDate,
    x.ipa_2_signatures AS IPA2Signatures,
    x.ipa_last_signature_date AS IPALastSignatureDate,
    x.ipa_due_diligence_min_review_date AS IPADueDiligenceMinReviewDate,
    COALESCE(x.cnt_offer_made, 0) AS OfferCount,
    x.unique_homes_offered AS UniqueHomesOffered,
    x.ipa_placement_admission_date AS PlannedPlacementStartDate,
    x.estimated_weekly_cost AS EstimatedWeeklyCost,
    CASE
      WHEN r.required_start_date IS NULL THEN 'Unspecified'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 1 THEN 'Critical'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 3 THEN 'High'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 7 THEN 'Medium'
      ELSE 'Planned'
    END AS PlacementUrgencyBand
  FROM referral_current r
  INNER JOIN referral_created c ON r.referral_id = c.referral_id
  LEFT JOIN referral_child child ON r.referral_id = child.referral_id
  LEFT JOIN closure_reason closure ON r.referral_id = closure.referral_id
  LEFT JOIN referral_enrichment x ON r.referral_id = x.referral_id
)
SELECT {AS_OF_SQL} AS AsOfDate,
  ReferralID, ChildID, ReferralCreatedDate, RequiredPlacementDate, ResponseRequiredDate,
  FirstActionDate, FirstOfferDate, OfferAcceptedDate, IPAIssuedDate,
  ReferralClosedDate, ReferralClosureReason, LastActivityDate, CurrentStatus,
  PlacementTypeRequired,
  CAST(NULL AS STRING) AS Region,
  PlacementUrgencyBand AS Priority,
  CAST(NULL AS STRING) AS ComplexityBand,
  PlacementUrgencyBand,
  CntOfferMade, FirstProviderSeenDate, IsNotSeenByProviders,
  IPAPlacementAdmissionDate, IPA2Signatures, IPALastSignatureDate,
  IPADueDiligenceMinReviewDate,
  CAST(NULL AS STRING) AS ChildCriticalityCode,
  OfferCount,
  COALESCE(UniqueHomesOffered, 0) AS UniqueHomesOffered,
  COALESCE(OfferCount, 0) > 0 AS HasOffer,
  DATEDIFF(TO_DATE(FirstActionDate), TO_DATE(ReferralCreatedDate)) AS DaysToFirstAction,
  DATEDIFF(TO_DATE(FirstOfferDate), TO_DATE(ReferralCreatedDate)) AS DaysToFirstOffer,
  DATEDIFF(TO_DATE(OfferAcceptedDate), TO_DATE(ReferralCreatedDate)) AS DaysToAcceptedOffer,
  DATEDIFF(TO_DATE(IPAIssuedDate), TO_DATE(ReferralCreatedDate)) AS DaysToIPA,
  DATEDIFF(COALESCE(TO_DATE(ReferralClosedDate), {AS_OF_SQL}),
    TO_DATE(ReferralCreatedDate)) AS DaysOpen,
  DATEDIFF({AS_OF_SQL}, TO_DATE(LastActivityDate)) AS DaysWithoutActivity,
  CASE WHEN RequiredPlacementDate IS NOT NULL AND RequiredPlacementDate < {AS_OF_SQL}
    THEN DATEDIFF({AS_OF_SQL}, RequiredPlacementDate) ELSE 0 END AS DaysPastRequiredDate,
  LOWER(COALESCE(CurrentStatus, '')) NOT IN
    ('closed','cancelled','withdrawn','completed') AS IsOpen,
  IPAIssuedDate IS NOT NULL AND RequiredPlacementDate IS NOT NULL
    AND TO_DATE(IPAIssuedDate) <= RequiredPlacementDate AS PlacedByRequiredDate,
  CASE
    WHEN IPAIssuedDate IS NOT NULL AND RequiredPlacementDate IS NOT NULL
      AND TO_DATE(IPAIssuedDate) <= RequiredPlacementDate THEN 'Placed by target'
    WHEN IPAIssuedDate IS NOT NULL THEN 'Placed after target'
    WHEN RequiredPlacementDate < {AS_OF_SQL} AND LOWER(COALESCE(CurrentStatus, '')) NOT IN
      ('closed','cancelled','withdrawn','completed') THEN 'Open overdue'
    WHEN LOWER(COALESCE(CurrentStatus, '')) NOT IN
      ('closed','cancelled','withdrawn','completed') THEN 'Open on track'
    ELSE 'Closed without placement'
  END AS RequiredPlacementDateOutcome,
  PlannedPlacementStartDate, EstimatedWeeklyCost,
  CURRENT_TIMESTAMP() AS GoldModelledAt
FROM base
WHERE TO_DATE(ReferralCreatedDate) <= {AS_OF_SQL}
""")


In [ ]:
snapshot = spark.table("gold.fact_referral").select(
    F.lit(AS_OF_DATE_VALUE).cast("date").alias("SnapshotDate"),
    "ReferralID", "ChildID", "ReferralCreatedDate", "RequiredPlacementDate",
    "FirstActionDate", "FirstOfferDate", "OfferAcceptedDate", "IPAIssuedDate",
    "ReferralClosedDate", "ReferralClosureReason", "CurrentStatus",
    "LastActivityDate", "PlacementTypeRequired", "Region", "Priority",
    "ComplexityBand", "PlacementUrgencyBand", "CntOfferMade", "FirstProviderSeenDate",
    "IsNotSeenByProviders", "IPAPlacementAdmissionDate", "IPA2Signatures",
    "IPALastSignatureDate", "IPADueDiligenceMinReviewDate",
    "IsOpen", "HasOffer", "OfferCount", "DaysOpen", "DaysWithoutActivity",
    "DaysPastRequiredDate", "PlacedByRequiredDate", "RequiredPlacementDateOutcome",
)
month_start = AS_OF_DATE_VALUE.replace(day=1)
next_month_start = (
    month_start.replace(year=month_start.year + 1, month=1)
    if month_start.month == 12
    else month_start.replace(month=month_start.month + 1)
)
month_predicate = (
    f"SnapshotDate >= DATE '{month_start.isoformat()}' AND "
    f"SnapshotDate < DATE '{next_month_start.isoformat()}'"
)
if not spark.catalog.tableExists(SNAPSHOT_TABLE):
    snapshot.write.format("delta").mode("overwrite").saveAsTable(SNAPSHOT_TABLE)
else:
    # Replacement is scoped to the active calendar month.
    (snapshot.write.format("delta").mode("overwrite")
        .option("replaceWhere", month_predicate)
        .option("mergeSchema", "true")
        .saveAsTable(SNAPSHOT_TABLE))
print(
    f"Snapshot refreshed for {AS_OF_DATE_VALUE}: {snapshot.count():,} referrals; "
    f"replaced active month {month_start:%Y-%m}"
)


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW gold.fact_referral_lifecycle_event AS
SELECT event_id AS EventID, referral_id AS ReferralID, event_type AS EventType,
  event_timestamp AS EventTimestamp, sequence_number AS SequenceNumber,
  created_by AS CreatedBy
FROM {EVENT_ROLLUP_SOURCE}
""")

# Source-grain Gold facts. These retain the individual offer, IPA placement
# and referral-provider records rather than collapsing them into FactReferral.
spark.sql(f"""
CREATE OR REPLACE VIEW gold.fact_offer AS
SELECT {AS_OF_SQL} AS AsOfDate,
  o.offer_id AS OfferID, rp.referral_id AS ReferralID,
  rp.provider_id AS ProviderID, o.provider_home_id AS HomeID,
  CAST(o.offer_date AS TIMESTAMP) AS OfferSubmittedDate,
  CAST(o.last_modified_date AS TIMESTAMP) AS OfferReviewedDate,
  CASE WHEN LOWER(COALESCE(o.offer_status, '')) IN
    ('offer_successful', 'offer_unsuccessful', 'offer_withdrawn',
     'accepted', 'approved', 'selected', 'declined', 'rejected', 'withdrawn')
    THEN CAST(o.last_modified_date AS TIMESTAMP) END AS OfferDecisionDate,
  o.offer_status AS OfferStatus, o.offer_type AS OfferType,
  CAST(o.estimated_start_date AS TIMESTAMP) AS ProposedPlacementStartDate,
  CAST(NULL AS INT) AS EstimatedDurationWeeks,
  CAST(COALESCE(o.core_weekly_fee, 0) + COALESCE(o.education_weekly_fee, 0)
    AS DECIMAL(19, 2)) AS EstimatedWeeklyCost,
  COALESCE(o.decline_reason_other, o.decline_reason, o.withdraw_reason)
    AS RejectionReason,
  o.child_summary_needs AS ChildSummaryNeeds,
  CAST(o.export_date AS TIMESTAMP) AS SourceExportDate,
  CURRENT_TIMESTAMP() AS GoldModelledAt
FROM silver.offer o
INNER JOIN silver.referral_provider rp
  ON o.referral_provider_id = rp.referral_provider_id
WHERE o.offer_date IS NULL OR TO_DATE(o.offer_date) <= {AS_OF_SQL}
""")

spark.sql(f"""
CREATE OR REPLACE VIEW gold.fact_placement AS
SELECT {AS_OF_SQL} AS AsOfDate,
  i.ipa_id AS PlacementID, i.referral_id AS ReferralID,
  i.offer_id AS AcceptedOfferID,
  CAST(i.created_datetime AS TIMESTAMP) AS IPAIssuedDate,
  CAST(i.placement_admission_date AS TIMESTAMP) AS PlannedPlacementStartDate,
  CAST(NULL AS TIMESTAMP) AS ActualPlacementStartDate,
  CAST(NULL AS TIMESTAMP) AS PlannedPlacementEndDate,
  CAST(NULL AS TIMESTAMP) AS ActualPlacementEndDate,
  CAST(i.placement_admission_date AS TIMESTAMP) AS PlacementAdmissionDate,
  CAST(i.costs_total_weekly_fee AS DECIMAL(19, 2)) AS EstimatedWeeklyCost,
  CAST(NULL AS DECIMAL(19, 2)) AS ActualWeeklyCost,
  i.status AS PlacementStatus, CAST(i.closed AS BOOLEAN) AS IsPlacementClosed,
  CAST(i.closed_datetime AS TIMESTAMP) AS PlacementEndedDate,
  CAST(NULL AS STRING) AS PlacementEndReason,
  CAST(i.export_date AS TIMESTAMP) AS SourceExportDate,
  CURRENT_TIMESTAMP() AS GoldModelledAt
FROM silver.ipa i
WHERE i.created_datetime IS NULL OR TO_DATE(i.created_datetime) <= {AS_OF_SQL}
""")

spark.sql(f"""
CREATE OR REPLACE VIEW gold.fact_referral_provider AS
SELECT {AS_OF_SQL} AS AsOfDate,
  rp.referral_provider_id AS ReferralProviderID, rp.referral_id AS ReferralID,
  rp.provider_id AS ProviderID, CAST(rp.export_date AS TIMESTAMP) AS FirstObservedDate,
  CAST(rp.is_excluded AS BOOLEAN) AS IsExcluded,
  CAST(rp.is_declined AS BOOLEAN) AS IsDeclined,
  CAST(rp.is_cancelled AS BOOLEAN) AS IsCancelled,
  CAST(rp.is_closed AS BOOLEAN) AS IsClosed,
  CASE
    WHEN rp.is_cancelled THEN 'Cancelled'
    WHEN rp.is_declined THEN 'Declined'
    WHEN rp.is_closed THEN 'Closed'
    WHEN rp.is_excluded THEN 'Excluded'
    ELSE 'Assigned'
  END AS ProviderResponseStatus,
  CURRENT_TIMESTAMP() AS GoldModelledAt
FROM silver.referral_provider rp
WHERE rp.export_date IS NULL OR TO_DATE(rp.export_date) <= {AS_OF_SQL}
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_kpi_referral_board_summary AS
SELECT AsOfDate, PlacementUrgencyBand, RequiredPlacementDateOutcome,
  COUNT(DISTINCT ReferralID) AS ReferralCount,
  SUM(CASE WHEN IsOpen THEN 1 ELSE 0 END) AS OpenReferralCount,
  SUM(CASE WHEN IsOpen AND RequiredPlacementDate < AsOfDate THEN 1 ELSE 0 END) AS OpenOverdueCount,
  SUM(CASE WHEN PlacedByRequiredDate THEN 1 ELSE 0 END) AS PlacedByRequiredDateCount,
  SUM(CASE WHEN HasOffer THEN 1 ELSE 0 END) AS ReferralsWithOfferCount,
  PERCENTILE_APPROX(DaysToIPA, 0.5) AS MedianDaysToIPA,
  SUM(COALESCE(EstimatedWeeklyCost, 0)) AS EstimatedWeeklyCost
FROM gold.fact_referral
GROUP BY AsOfDate, PlacementUrgencyBand, RequiredPlacementDateOutcome
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_kpi_referral_monthly AS
SELECT DATE_TRUNC('month', ReferralCreatedDate) AS ReferralCreatedMonth,
  COUNT(DISTINCT ReferralID) AS NewReferralCount,
  SUM(CASE WHEN HasOffer THEN 1 ELSE 0 END) AS ReferralsWithOfferCount,
  SUM(CASE WHEN IPAIssuedDate IS NOT NULL THEN 1 ELSE 0 END) AS IPACount,
  SUM(CASE WHEN PlacedByRequiredDate THEN 1 ELSE 0 END) AS PlacedByRequiredDateCount,
  SUM(CASE WHEN IsOpen THEN 1 ELSE 0 END) AS OpenReferralCount
FROM gold.fact_referral
GROUP BY DATE_TRUNC('month', ReferralCreatedDate)
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_provider_offer_performance AS
SELECT rp.provider_id AS ProviderID,
  COUNT(DISTINCT rp.referral_id) AS ReferralsReceived,
  COUNT(DISTINCT o.offer_id) AS OffersSubmitted,
  COUNT(DISTINCT CASE WHEN LOWER(o.offer_status) IN ('accepted','approved','selected')
    THEN o.offer_id END) AS OffersAccepted,
  COUNT(DISTINCT CASE WHEN f.PlacedByRequiredDate THEN f.ReferralID END) AS ReferralsPlacedByTarget
FROM silver.referral_provider rp
LEFT JOIN silver.offer o ON rp.referral_provider_id = o.referral_provider_id
LEFT JOIN gold.fact_referral f ON rp.referral_id = f.ReferralID
GROUP BY rp.provider_id
""")
